# Verificacion - Spark lee tu datalake
**Curso:** ST1630-2026-2 · **Semana:** S4-S5
**Equipo:** Andrés Felipe Vélez (afveleza@eafit.edu.co), Samuel Samper Cardona (ssamperc@eafit.edu.co), Sebastián Salazar Henao (salazarh3@eafit.edu.co), Hellen Yanes Doria (hyanesd@eafit.edu.co)
**Fecha:** 2026-08-13
**Fecha:** 2026-08-13

## Objetivo

Cerrar el Lab 1a confirmando que tu cluster EMR puede leer el datalake
que construiste (Partes 1-4): conectar Spark a tu bucket S3, leer el
archivo Parquet de Bronze, y repetir el benchmark Parquet vs. CSV visto
en la clase de S4.

**Que debe verse al final para confirmar que el lab esta completo:**
- La Celda 2 muestra el schema y 5 filas del Parquet leido desde S3
  (si esto funciona, tu bucket, tu rol IAM y tu cluster estan bien
  configurados de punta a punta).
- La Celda 3 imprime el tiempo de una misma consulta en Parquet y en
  CSV, y el ratio entre ambos.
- Completaste el analisis de la Celda 4 y capturaste el DAG de Spark UI
  como indica la Celda 5.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# getOrCreate() funciona tanto en EMR (donde ya existe una sesión activa
# administrada por el clúster) como en un entorno local con pyspark
# instalado, sin necesitar ramas de código distintas.
spark = SparkSession.builder.appName("ST1630-Lab1a-Verificacion").getOrCreate()

# EDITAR: reemplaza por el bucket que creaste en setup_s3.sh
# (convención: st1630-{tu-usuario}-{año})
BUCKET = "st1630-afveleza-2026"

# En EMR, S3 se referencia directamente con el esquema s3://.
# En local (con las credenciales de AWS Academy exportadas), la misma
# ruta también funciona porque Spark usa el conector S3A por debajo.
ruta_parquet = f"s3://{BUCKET}/bronze/ventas/prueba_parquet.parquet"
ruta_csv = f"s3://{BUCKET}/bronze/ventas/prueba_csv.csv"

df_parquet = spark.read.parquet(ruta_parquet)

df_parquet.printSchema()
df_parquet.show(5, truncate=False)

# Si ves el schema y las filas de arriba, tu datalake funciona
# correctamente de punta a punta: bucket, permisos IAM y clúster EMR.
print("Filas leídas:", df_parquet.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 23:55:54 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


root
 |-- order_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- region: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precio_unit: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- devuelto: boolean (nullable = true)



+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
|order_id  |fecha     |region      |producto |categoria  |cantidad|precio_unit|total    |canal |devuelto|
+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
|ORD-000001|2026-04-22|Cali        |Mouse    |Electrónica|1       |789300.0   |789300.0 |online|false   |
|ORD-000002|2026-03-22|Barranquilla|Gorra    |Ropa       |2       |57000.0    |114000.0 |tienda|false   |
|ORD-000003|2026-01-06|Bogotá      |Zapatos  |Ropa       |4       |163800.0   |655200.0 |online|false   |
|ORD-000004|2025-11-26|Barranquilla|Panela   |Alimentos  |3       |54900.0    |164700.0 |online|false   |
|ORD-000005|2026-01-07|Cali        |Audífonos|Electrónica|4       |251100.0   |1004400.0|tienda|true    |
+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
only showing top 5 rows

Filas leídas: 10000


In [2]:
import time

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta_csv)

# Misma consulta sobre ambos formatos: filtrar por región y categoría,
# y agregar el total vendido -- el tipo de consulta selectiva que se
# beneficia de predicado pushdown y column pruning en formatos
# columnares (visto en la clase de S4, slide de Parquet vs. CSV).

def benchmark(df, nombre):
    inicio = time.time()
    resultado = (
        df.filter((F.col("region") == "Bogotá") & (F.col("categoria") == "Electrónica"))
          .groupBy("producto")
          .agg(F.sum("total").alias("total_vendido"))
          .orderBy(F.col("total_vendido").desc())
          .collect()  # acción -- fuerza la ejecución real, no solo el plan
    )
    duracion = time.time() - inicio
    print(f"{nombre}: {duracion:.3f} s ({len(resultado)} filas de resultado)")
    return duracion

tiempo_parquet = benchmark(df_parquet, "Parquet")
tiempo_csv = benchmark(df_csv, "CSV")

ratio = tiempo_csv / tiempo_parquet if tiempo_parquet > 0 else float("inf")
print(f"\nRatio CSV / Parquet: {ratio:.2f}x")

Parquet: 1.834 s (6 filas de resultado)
CSV: 0.782 s (6 filas de resultado)

Ratio CSV / Parquet: 0.43x


## Analisis - completa antes de entregar

### a) Tamano en disco

Cuanto pesa prueba_parquet.parquet frente a prueba_csv.csv?

**Respuesta:** prueba_parquet.parquet pesa 185.4 KiB y prueba_csv.csv pesa 798.3 KiB. El Parquet ocupa aproximadamente 4.3 veces menos espacio que el CSV para el mismo conjunto de 10.000 filas. Esto se debe a que Parquet usa codificacion columnar con compresion integrada (Snappy por defecto), eliminando redundancia dentro de cada columna, mientras que CSV almacena cada valor como texto plano sin compresion.

### b) Tiempo de la consulta

Cuanto tardo la consulta de la Celda 3 en cada formato?

**Respuesta:** La consulta (filtrar region=Bogota y categoria=Electronica, agrupar por producto, sumar total) tardo:
- Parquet: 1.834 s
- CSV: 0.782 s

### c) Ratio de mejora

Cual fue el ratio de mejora (CSV / Parquet) obtenido?

**Respuesta:** El ratio obtenido fue 0.43x, lo que significa que en esta ejecucion el CSV fue mas rapido que el Parquet, resultado opuesto al ~9x esperado en clase. Esto se explica por tres factores:
1. Dataset pequeno (10.000 filas): la ventaja de Parquet (predicado pushdown y column pruning) se manifiesta con volumenes grandes. Con 10K filas el overhead de inicializar el lector columnar supera el beneficio.
2. Cluster minimo (2 nodos m5.xlarge): sin multiples particiones fisicas que explotar en paralelo, el column pruning de Parquet no genera ganancia real de I/O distribuido.
3. Parquet fue la primera lectura en frio desde S3; el CSV ya tenia datos parcialmente en cache de la JVM.
En produccion con cientos de GB y consultas selectivas sobre pocas columnas, la ventaja de Parquet seria claramente visible (~9x o mas).

### d) Conexion con el Teorema CAP

Por que S3 con replicacion entre multiples zonas de disponibilidad es una decision CP dentro del Teorema CAP?

**Respuesta:** S3 es CP (Consistencia + Tolerancia a particiones) porque garantiza consistencia de lectura tras escritura: una vez confirmada una escritura (replicada durablemente en multiples zonas), cualquier lectura posterior siempre vera la version mas reciente. Esta garantia se mantiene ante particiones de red entre zonas.
Lo que S3 sacrifica es disponibilidad de escritura: ante una particion de red, S3 prefiere rechazar o retardar una escritura antes que confirmarla sin haberla replicado completamente. Un sistema puramente AP (como Cassandra con consistency level ONE) confirmaria la escritura aunque quede temporalmente inconsistente entre replicas. S3 elige consistencia a costa de una disponibilidad de escritura levemente menor.

## Captura del DAG en Spark UI

1. En EMR Studio (o en la consola de tu clúster), abre **Spark UI /
   History Server**.
2. Busca el job correspondiente a la Celda 3 (el `groupBy` + `agg` +
   `orderBy` sobre el Parquet).
3. Abre la pestaña **SQL / DataFrame** y captura una imagen del plan
   (o del DAG visual) que incluya al menos un nodo **Exchange**.
4. Guarda la captura como `dag_spark_ui.png` dentro de tu carpeta de
   entrega y referencíala en tu PR.

**Verifica:** la captura debe mostrar el nombre de tu aplicación
(`ST1630-Lab1a-Verificacion`, definido en la Celda 2) para que quede
claro que es tu propia ejecución.

## Bitacora de delegacion

Completa segun lo que realmente delegaste a un agente de IA durante este notebook.

| Tarea | Delegado a agente? | Herramienta | Justificacion |
|---|---|---|---|
| Boilerplate de SparkSession / lectura de S3 | no | - | El codigo ya estaba provisto en el notebook; solo edite la variable BUCKET. |
| Diseno de la consulta del benchmark (Celda 3) | no | - | La consulta (filter + groupBy + agg + orderBy) ya estaba implementada en el notebook del lab. |
| Interpretacion de los resultados (Celda 4) | parcial | Antigravity (Gemini) | El agente ayudo a identificar los valores reales de tiempo y tamano desde los outputs; la justificacion del ratio invertido y la conexion con CAP son razonamiento propio. |
| Troubleshooting de errores de conexion a S3 | si | Antigravity (Gemini) | El agente diagnostico y resolvio los problemas de permisos del .pem en Windows (CRLF vs LF, icacls) y el PATH de jupyter en el cluster EMR. |

> Recuerda: la interpretacion de los resultados y la conexion con CAP (pregunta d) deben reflejar tu propio razonamiento.